In [ ]:
import pandas as pd
from google.colab import drive
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

# Mount Google Drive
# drive.mount('/content/drive')

train_path = '/kaggle/input/titanic/train.csv'
test_path = '/kaggle/input/titanic/test.csv'

train_data = pd.read_csv(train_path)
test_data = pd.read_csv(test_path)

train_clean = train_data.copy()
test_clean = test_data.copy()

le_sex = LabelEncoder()
le_Embarked = LabelEncoder()

# Preprocess train data
columns_to_drop = ['PassengerId', 'Name', 'SibSp', 'Parch', 'Ticket', 'Cabin']

# MENGUBAH NILAI YANG BUKAN INT MENJADI INT
train_clean['Sex'] = le_sex.fit_transform(train_clean['Sex'])
train_clean['Embarked'] = le_Embarked.fit_transform(train_clean['Embarked'])

# MENGISI NILAI YANG KOSONG DENGAN MENGGUNAKAN MEDIAN, BISA JUGA MEAN
train_clean['Fare'] = SimpleImputer(strategy='median').fit_transform(train_clean[['Fare']])
train_clean['Age'] = SimpleImputer(strategy='median').fit_transform(train_clean[['Age']])

# MENGHAPUS KOLOM(FITUR) YANG TIDAK DIPERLUKAN
train_clean = train_clean.drop(columns=columns_to_drop)

# MEMISAHKAN X, y train
y_train = train_clean['Survived']
X_train = train_clean.drop('Survived', axis=1)

# Preprocess test data
test_clean['Sex'] = le_sex.fit_transform(test_clean['Sex'])
test_clean['Embarked'] = le_Embarked.fit_transform(test_clean['Embarked'])

test_clean['Fare'] = SimpleImputer(strategy='median').fit_transform(test_clean[['Fare']])
test_clean['Age'] = SimpleImputer(strategy='median').fit_transform(test_clean[['Age']])

X_test = test_clean.drop(columns=columns_to_drop)

# Training the model
model = RandomForestClassifier(n_estimators=150, max_depth=4, random_state=1)
model.fit(X_train, y_train)

# MAKING PREDICTIONS
y_train_pred = model.predict(X_train)
y_pred = model.predict(X_test)

# Accuracy on training data
train_accuracy_score = accuracy_score(y_train, y_train_pred)
print(f"Training Accuracy: {train_accuracy_score:.4f}")

# Buat output submission
output = pd.DataFrame({'PassengerId': test_clean['PassengerId'], 'Survived': y_pred})
output.to_csv('submission.csv', index=False)